# LLM-Judge Human Validation

3 annotators who are 4th-year medical students.

This notebook does the following:
- Human agreement
    - Cohen’s kappa (pairwise)
    - Fleiss's kappa
    - Krippendorff's alpha
- LLM accuracy & aligment with human judgments
    - 95% CI for accuracy
    - Macro/weighted F1
    - Per-class metrics
    - Confusion matrix
    - LLM vs each annotator agreement

In [12]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score
)

from statsmodels.stats.inter_rater import fleiss_kappa
from statsmodels.stats.proportion import proportion_confint

from itertools import combinations

import krippendorff

In [ ]:
# LOAD DATA

# ------------------------------------------------------------
# CSV format:
# item_id,annotator_1,annotator_2,annotator_3,llm
# ------------------------------------------------------------

df = pd.read_csv("../data/annotations.csv")

ANNOTATORS = ["annotator_1", "annotator_2", "annotator_3"]
LLM_COL = "llm"

In [14]:
LABELS = ["higher", "lower", "same", "uncertain"]
LABEL_MAP = {
    "higher": 0,
    "lower": 1,
    "same": 2,
    "uncertain": 3
}


In [15]:
# PAIRWISE HUMAN AGREEMENT

for a1, a2 in combinations(ANNOTATORS, 2):

    agreement = (df[a1] == df[a2]).mean()

    kappa = cohen_kappa_score(
        df[a1],
        df[a2],
        labels=LABELS
    )

    print(f"\n{a1} vs {a2}")
    print(f"Raw Agreement: {agreement:.3f}")
    print(f"Cohen's Kappa: {kappa:.3f}")


annotator_1 vs annotator_2
Raw Agreement: 0.675
Cohen's Kappa: 0.570

annotator_1 vs annotator_3
Raw Agreement: 0.700
Cohen's Kappa: 0.598

annotator_2 vs annotator_3
Raw Agreement: 0.925
Cohen's Kappa: 0.897


In [16]:
# FLEISS' KAPPA

# Convert annotations into Fleiss matrix format
# Each row = one item
# Each column = count of votes per class

fleiss_matrix = []

for _, row in df.iterrows():

    counts = []

    for label in LABELS:
        count = sum(row[a] == label for a in ANNOTATORS)
        counts.append(count)

    fleiss_matrix.append(counts)

fleiss_matrix = np.array(fleiss_matrix)

fk = fleiss_kappa(fleiss_matrix)

print(f"Fleiss' Kappa: {fk:.3f}")

Fleiss' Kappa: 0.684


In [17]:
# KRIPPENDORFF'S ALPHA

# krippendorff expects:
# rows = annotators
# cols = items


kripp_data = (
    df[ANNOTATORS]
    .apply(lambda col: col.map(LABEL_MAP))
    .T
    .values
)
# kripp_data = df[ANNOTATORS].T.values

alpha = krippendorff.alpha(
    reliability_data=kripp_data,
    level_of_measurement='nominal'
)

print(f"Krippendorff's Alpha: {alpha:.3f}")

Krippendorff's Alpha: 0.687


In [18]:
# MAJORITY VOTE LABELS

def majority_vote(row):

    counts = row.value_counts()

    # Majority exists
    if counts.iloc[0] >= 2:
        return counts.idxmax()

    # No majority (all disagree)
    return np.nan

df["majority_label"] = df[ANNOTATORS].apply(
    majority_vote,
    axis=1
)

n_total = len(df)
n_majority = df["majority_label"].notna().sum()

print(f"Majority labels available for {n_majority}/{n_total} items")

# Remove rows with no majority label
eval_df = df.dropna(subset=["majority_label"]).copy()

Majority labels available for 40/40 items


In [24]:
# LLM PERFORMANCE AGAINST MAJORITY LABELS

y_true = eval_df["majority_label"]
y_pred = eval_df[LLM_COL]

# Accuracy
acc = accuracy_score(y_true, y_pred)

# 95% CI for accuracy
ci_low, ci_high = proportion_confint(
    count=(y_true == y_pred).sum(),
    nobs=len(y_true),
    alpha=0.05,
    method='wilson'
)

# Macro F1
macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="macro"
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    labels=LABELS,
    average="weighted"
)

# Cohnen's Kappa between LLM and majority vote
kappa = cohen_kappa_score(
    y_true,
    y_pred,
    labels=LABELS
)

print(f"Accuracy: {acc:.3f}")
print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")

print(f"Macro F1: {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")

print(f"Cohen's Kappa: {kappa:.3f}")

Accuracy: 0.675
95% CI: [0.520, 0.799]
Macro F1: 0.616
Weighted F1: 0.650
Cohen's Kappa: 0.530


In [20]:
# PER-CLASS METRICS

report = classification_report(
    y_true,
    y_pred,
    labels=LABELS,
    digits=3
)

print(report)

              precision    recall  f1-score   support

      higher      0.667     0.333     0.444         6
       lower      0.889     0.727     0.800        11
        same      1.000     0.333     0.500         9
   uncertain      0.560     1.000     0.718        14

    accuracy                          0.675        40
   macro avg      0.779     0.598     0.616        40
weighted avg      0.765     0.675     0.650        40



In [21]:
# CONFUSION MATRIX

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=LABELS
)

cm_df = pd.DataFrame(
    cm,
    index=[f"TRUE_{x}" for x in LABELS],
    columns=[f"PRED_{x}" for x in LABELS]
)

print(cm_df)

                PRED_higher  PRED_lower  PRED_same  PRED_uncertain
TRUE_higher               2           0          0               4
TRUE_lower                0           8          0               3
TRUE_same                 1           1          3               4
TRUE_uncertain            0           0          0              14


In [22]:
# LLM VS EACH ANNOTATOR

for annotator in ANNOTATORS:

    acc = accuracy_score(
        df[annotator],
        df[LLM_COL]
    )

    macro_f1 = f1_score(
        df[annotator],
        df[LLM_COL],
        labels=LABELS,
        average="macro"
    )

    kappa = cohen_kappa_score(
        df[annotator],
        df[LLM_COL],
        labels=LABELS
    )

    print(f"\nLLM vs {annotator}")
    print(f"Accuracy: {acc:.3f}")
    print(f"Macro F1: {macro_f1:.3f}")
    print(f"Cohen's Kappa: {kappa:.3f}")


LLM vs annotator_1
Accuracy: 0.450
Macro F1: 0.427
Cohen's Kappa: 0.282

LLM vs annotator_2
Accuracy: 0.725
Macro F1: 0.653
Cohen's Kappa: 0.590

LLM vs annotator_3
Accuracy: 0.650
Macro F1: 0.598
Cohen's Kappa: 0.504
